# 01 · ETL y calidad

Evidencia de extracción, normalización, reglas de calidad y reconciliación de persistencia.

**Reproducibilidad:** ejecutar desde la raíz del repositorio después del paso correspondiente de `scripts/run_all.py`.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
print(f'Proyecto detectado: {ROOT.name}')

Proyecto detectado: adidas-dest-ac


In [2]:
import json
import pandas as pd
orders = pd.read_parquet(ROOT/'data/processed/orders_enriched.parquet')
quality = pd.read_csv(ROOT/'data/quality/data_quality_report.csv')
validation = json.loads((ROOT/'data/quality/persistence_validation.json').read_text(encoding='utf-8'))
validation['source_counts'], orders.shape

({'products': 120, 'users': 100, 'carts': 200}, (733, 48))

In [3]:
orders[['order_id','user_id','product_id','country','category','quantity','unit_price','revenue','data_quality_flag']].head(10)

,order_id,user_id,product_id,country,category,quantity,unit_price,revenue,data_quality_flag
0,1,65,8,Argentina,Footwear,18.0,91.60,1648.80,OK
1,1,65,58,Argentina,Apparel,112.0,87.71,9823.52,OK
2,2,6,91,Chile,Equipment,61.0,36.90,2250.90,OK
3,2,6,57,Chile,Apparel,83.0,64.26,5333.58,OK
4,3,87,7,Chile,Footwear,13.0,203.78,2649.14,OK
5,3,87,102,Chile,Equipment,76.0,89.84,6827.84,OK
6,3,87,81,Chile,Accessories,41.0,29.43,1206.63,OK
7,3,87,83,Chile,Accessories,114.0,75.79,8640.06,OK
8,3,87,43,Chile,Apparel,82.0,24.85,2037.70,OK
9,4,45,84,Perú,Accessories,55.0,51.74,2845.70,OK


In [4]:
quality.sort_values('registros_afectados', ascending=False).head(10)

,regla_evaluada,registros_revisados,registros_afectados,porcentaje_afectado,accion_aplicada,justificacion,severidad
13,Revenue extremo por IQR,733,44,6.00,Conservar y monitorear,"En mayoristas, valores altos pueden ser compra...",Media
16,Calificación nula,733,18,2.46,Conservar nulo,No se inventa percepción de producto.,Baja
2,Cantidad nula,733,12,1.64,Conservar; revenue queda nulo,Imputar cantidad inventaría ventas no observadas.,Alta
15,Inventario nulo,733,8,1.09,Conservar nulo,No afecta el cálculo histórico de ingresos.,Baja
1,Identificador faltante,733,0,0.00,Conservar y marcar,La trazabilidad incompleta impide uniones conf...,Alta
0,Línea duplicada,733,0,0.00,Conservar y marcar,La clave compuesta debe ser única; no se elimi...,Alta
6,País inconsistente,733,0,0.00,Normalizar variantes conocidas,Mantiene comparabilidad geográfica sin inventa...,Media
3,Cantidad no positiva,733,0,0.00,Conservar y excluir de KPIs,No representa una venta válida sin nota de cré...,Alta
4,Precio nulo,733,0,0.00,Conservar; revenue queda nulo,Se prioriza el precio histórico de la línea.,Alta
5,Precio no positivo,733,0,0.00,Conservar y excluir de KPIs,Un precio no positivo requiere revisión de neg...,Alta


## Conclusión

Los conteos de API, CSV, Parquet y SQLite reconcilian. Las cantidades nulas se conservan y el revenue correspondiente permanece nulo; los outliers se marcan sin eliminarlos.